# Implementación de modelos de Redes Neuronales Profundas

En este archivo, voy a poner en práctica lo aprendido a lo largo de la asignatura tanto en la parte de teoría como en la parte de práctica. La idea es crear una CNN capaz de analizar radiografías de tórax y clasificarlas en tres categorías: pacientes sanos, con neumonía bacteriana y con neumonía vírica.

**Realizado por Rodrigo Gálvez Travalja**

---

## Estructura del notebook
1. Instalación e importación de librerías
2. Comprobación de GPU
3. Descarga del dataset
4. Preprocesamiento e imágenes
5. Definición de la CNN
6. Entrenamiento y validación
7. Predicción y visualización
8. Guardado del modelo

## 1. Instalación e importación de librerías

In [ ]:
%pip install torch torchvision kagglehub matplotlib pandas scikit-learn --quiet

import os
import glob
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms.functional import to_pil_image
from PIL import Image
from sklearn.model_selection import train_test_split

import kagglehub

print('Librerías importadas correctamente.')

## 2. Comprobación de GPU y configuración del device

In [ ]:
# Verificar disponibilidad de CUDA y seleccionar device
print(f'Versión de PyTorch: {torch.__version__}')
print(f'CUDA disponible:    {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU:                {torch.cuda.get_device_name(0)}')
    print(f'Versión CUDA:       {torch.version.cuda}')

# Variable global de device: usada en todo el notebook
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nUsando device: {device}')

## 3. Descarga del dataset desde Kaggle

In [ ]:
# Descarga del dataset Chest X-Ray Images (Pneumonia)
ruta = kagglehub.dataset_download('paultimothymooney/chest-xray-pneumonia')
print(f'Dataset descargado en: {ruta}')

# Rutas a los splits del dataset
ruta_datos    = os.path.join(ruta, 'chest_xray')
RUTA_TRAIN    = os.path.join(ruta_datos, 'train')
RUTA_VAL      = os.path.join(ruta_datos, 'val')
RUTA_TEST     = os.path.join(ruta_datos, 'test')   # minúsculas: Linux es case-sensitive

# Verificar que las carpetas existen
for nombre, ruta_split in [('train', RUTA_TRAIN), ('val', RUTA_VAL), ('test', RUTA_TEST)]:
    existe = os.path.isdir(ruta_split)
    print(f'  {nombre}: {ruta_split} → {"OK" if existe else "NO ENCONTRADA"}')

## 4. Preprocesamiento e imágenes

El dataset de Kaggle organiza las imágenes en dos carpetas: `NORMAL` y `PNEUMONIA`.
Dentro de `PNEUMONIA`, el nombre del fichero indica si es bacteriana (`bacteria`) o vírica (`virus`).
Creamos un Dataset personalizado que lee esta estructura y asigna las etiquetas correctas para la clasificación en 3 clases.

In [ ]:
# Tamaño de imagen y valores de normalización ImageNet
TAMANIO_IMAGEN = (100, 100)
IMAGENET_MEAN  = [0.485, 0.456, 0.406]
IMAGENET_STD   = [0.229, 0.224, 0.225]

# Nombres de clases — deben coincidir con architecture.py
NOMBRES_CLASES = ['NORMAL', 'PNEUMONIA_BACTERIAL', 'PNEUMONIA_VIRAL']

# Transformaciones para entrenamiento (con data augmentation)
transformacion_entrenamiento = transforms.Compose([
    transforms.Resize(TAMANIO_IMAGEN),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Transformaciones para validación y test (sin augmentation, solo normalización)
transformacion_val_test = transforms.Compose([
    transforms.Resize(TAMANIO_IMAGEN),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])


class DatasetRayosX(Dataset):
    """
    Dataset personalizado para radiografías de tórax con 3 clases.
    Lee la estructura de carpetas NORMAL / PNEUMONIA del dataset de Kaggle
    y distingue neumonía bacteriana y vírica por el nombre del fichero.
    """

    # Mapeo de clase a índice numérico
    MAPA_ETIQUETAS = {
        'NORMAL':              0,
        'PNEUMONIA_BACTERIAL': 1,
        'PNEUMONIA_VIRAL':     2
    }

    def __init__(self, directorio_raiz, transformacion=None):
        self.rutas_imagenes = []
        self.etiquetas      = []
        self.transformacion = transformacion

        # Imágenes normales
        for ruta_img in glob.glob(os.path.join(directorio_raiz, 'NORMAL', '*.jpeg')):
            self.rutas_imagenes.append(ruta_img)
            self.etiquetas.append(self.MAPA_ETIQUETAS['NORMAL'])

        # Imágenes de neumonía — distinguir bacteriana vs vírica por nombre de fichero
        for ruta_img in glob.glob(os.path.join(directorio_raiz, 'PNEUMONIA', '*.jpeg')):
            nombre = os.path.basename(ruta_img).lower()
            if 'bacteria' in nombre:
                self.rutas_imagenes.append(ruta_img)
                self.etiquetas.append(self.MAPA_ETIQUETAS['PNEUMONIA_BACTERIAL'])
            elif 'virus' in nombre:
                self.rutas_imagenes.append(ruta_img)
                self.etiquetas.append(self.MAPA_ETIQUETAS['PNEUMONIA_VIRAL'])

        print(f'Dataset cargado desde: {directorio_raiz}')
        print(f'  NORMAL:               {self.etiquetas.count(0)}')
        print(f'  PNEUMONIA_BACTERIAL:  {self.etiquetas.count(1)}')
        print(f'  PNEUMONIA_VIRAL:      {self.etiquetas.count(2)}')
        print(f'  Total:                {len(self.rutas_imagenes)}')

    def __len__(self):
        return len(self.rutas_imagenes)

    def __getitem__(self, idx):
        imagen  = Image.open(self.rutas_imagenes[idx]).convert('RGB')
        etiqueta = self.etiquetas[idx]
        if self.transformacion:
            imagen = self.transformacion(imagen)
        return imagen, etiqueta


# Crear datasets
dataset_train_completo = DatasetRayosX(RUTA_TRAIN, transformacion=transformacion_entrenamiento)
dataset_test           = DatasetRayosX(RUTA_TEST,  transformacion=transformacion_val_test)

# Dividir train en train (80%) y validación (20%) con stratify para mantener proporción de clases
indices_train, indices_val = train_test_split(
    list(range(len(dataset_train_completo))),
    test_size=0.2,
    stratify=dataset_train_completo.etiquetas,
    random_state=42
)

dataset_train = data.Subset(dataset_train_completo, indices_train)
dataset_val   = data.Subset(dataset_train_completo, indices_val)

# DataLoaders
loader_train = DataLoader(dataset_train, batch_size=32, shuffle=True,  num_workers=2)
loader_val   = DataLoader(dataset_val,   batch_size=32, shuffle=False, num_workers=2)
loader_test  = DataLoader(dataset_test,  batch_size=32, shuffle=False, num_workers=2)

print(f'\nBatches de entrenamiento: {len(loader_train)}')
print(f'Batches de validación:    {len(loader_val)}')
print(f'Batches de test:          {len(loader_test)}')

## 5. Definición de la CNN

Arquitectura de 3 bloques convolucionales seguidos de capas fully connected.
Cada bloque aplica Conv2d + ReLU + MaxPool2d(2,2), reduciendo la resolución a la mitad.
Con input 100×100: 100 → 50 → 25 → 12, por lo que la capa FC recibe 128 × 12 × 12 = 18432 características.

In [ ]:
class ModeloCNN(nn.Module):
    """
    Red neuronal convolucional para clasificación de radiografías de tórax.

    Arquitectura:
        Conv2d(3→32)  + ReLU + MaxPool2d(2,2)
        Conv2d(32→64) + ReLU + MaxPool2d(2,2)
        Conv2d(64→128)+ ReLU + MaxPool2d(2,2)
        Dropout(0.5)
        Linear(128*12*12 → 128) + ReLU
        Linear(128 → num_clases)
    """

    def __init__(self, num_clases: int = 3):
        super(ModeloCNN, self).__init__()
        self.conv1   = nn.Conv2d(3,   32,  kernel_size=3, padding=1)
        self.conv2   = nn.Conv2d(32,  64,  kernel_size=3, padding=1)
        self.conv3   = nn.Conv2d(64,  128, kernel_size=3, padding=1)
        self.pool    = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.5)
        self.fc1     = nn.Linear(128 * 12 * 12, 128)
        self.fc2     = nn.Linear(128, num_clases)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = self.pool(torch.relu(self.conv3(x)))
        x = self.dropout(x)
        x = x.view(x.size(0), -1)          # Aplanar
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)                     # Logits — softmax se aplica fuera
        return x


# Instanciar modelo y moverlo al device (GPU si está disponible)
modelo = ModeloCNN(num_clases=3).to(device)
print(modelo)

# Contar parámetros entrenables
total_params = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
print(f'\nParámetros entrenables: {total_params:,}')

## 6. Entrenamiento y validación

Criterio de parada: loss de entrenamiento ≤ 0.2.

In [ ]:
def entrenar_modelo(modelo, loader_train, loader_val, num_epocas=20, lr=0.001):
    """
    Entrena el modelo y evalúa en validación cada época.
    Se detiene si el loss de entrenamiento baja de 0.2.

    Returns:
        modelo entrenado y listas de métricas por época.
    """
    optimizador = optim.RMSprop(modelo.parameters(), lr=lr)
    criterio    = nn.CrossEntropyLoss()

    perdidas_train,  perdidas_val   = [], []
    precisiones_train, precisiones_val = [], []

    for epoca in range(num_epocas):

        # --- Fase de entrenamiento ---
        modelo.train()
        perdida_acum = 0.0
        correctos    = 0
        total        = 0

        for entradas, etiquetas in loader_train:
            # Mover datos al device (GPU/CPU)
            entradas = entradas.to(device)
            etiquetas = etiquetas.to(device)

            optimizador.zero_grad()
            salidas = modelo(entradas)
            perdida = criterio(salidas, etiquetas)
            perdida.backward()
            optimizador.step()

            perdida_acum += perdida.item()
            _, predicho   = torch.max(salidas, 1)
            correctos     += (predicho == etiquetas).sum().item()
            total         += etiquetas.size(0)

        loss_train = perdida_acum / len(loader_train)
        acc_train  = 100 * correctos / total
        perdidas_train.append(loss_train)
        precisiones_train.append(acc_train)

        # --- Fase de validación ---
        modelo.eval()
        perdida_val_acum = 0.0
        correctos_val    = 0
        total_val        = 0

        with torch.no_grad():
            for entradas, etiquetas in loader_val:
                entradas  = entradas.to(device)
                etiquetas = etiquetas.to(device)

                salidas = modelo(entradas)
                perdida = criterio(salidas, etiquetas)

                perdida_val_acum += perdida.item()
                _, predicho       = torch.max(salidas, 1)
                correctos_val    += (predicho == etiquetas).sum().item()
                total_val        += etiquetas.size(0)

        loss_val = perdida_val_acum / len(loader_val)
        acc_val  = 100 * correctos_val / total_val
        perdidas_val.append(loss_val)
        precisiones_val.append(acc_val)

        print(f'Época {epoca+1:02d}/{num_epocas} | '
              f'Loss train: {loss_train:.4f}  Acc train: {acc_train:.2f}% | '
              f'Loss val: {loss_val:.4f}  Acc val: {acc_val:.2f}%')

        # Criterio de parada: loss de entrenamiento <= 0.2
        if loss_train <= 0.2:
            print(f'\nCriterio de parada alcanzado en época {epoca+1}: loss train = {loss_train:.4f} ≤ 0.2')
            break

    return modelo, perdidas_train, perdidas_val, precisiones_train, precisiones_val


# Ejecutar entrenamiento
modelo, perdidas_train, perdidas_val, precisiones_train, precisiones_val = entrenar_modelo(
    modelo, loader_train, loader_val, num_epocas=20, lr=0.001
)

## 7. Visualización de métricas de entrenamiento

In [ ]:
def mostrar_graficas(perdidas_train, perdidas_val, precisiones_train, precisiones_val):
    """Muestra las gráficas de loss y accuracy para train y validación."""
    epocas = range(1, len(perdidas_train) + 1)

    plt.figure(figsize=(14, 5))

    # Gráfica de pérdida
    plt.subplot(1, 2, 1)
    plt.plot(epocas, perdidas_train, label='Entrenamiento')
    plt.plot(epocas, perdidas_val,   label='Validación')
    plt.axhline(y=0.2, color='r', linestyle='--', label='Criterio parada (0.2)')
    plt.title('Pérdida por época')
    plt.xlabel('Época')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    # Gráfica de precisión
    plt.subplot(1, 2, 2)
    plt.plot(epocas, precisiones_train, label='Entrenamiento')
    plt.plot(epocas, precisiones_val,   label='Validación')
    plt.title('Precisión por época')
    plt.xlabel('Época')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()


mostrar_graficas(perdidas_train, perdidas_val, precisiones_train, precisiones_val)

## 8. Predicción sobre el conjunto de test

In [ ]:
def predecir_con_probabilidades(modelo, loader_test, nombres_clases, max_imagenes=5):
    """
    Muestra imágenes del conjunto de test con sus probabilidades por clase
    ordenadas de mayor a menor.
    """
    modelo.eval()
    mostradas = 0

    with torch.no_grad():
        for entradas, _ in loader_test:
            entradas = entradas.to(device)
            salidas  = modelo(entradas)
            probs    = F.softmax(salidas, dim=1).cpu().numpy()

            for i in range(entradas.size(0)):
                if mostradas >= max_imagenes:
                    return

                prob_imagen  = probs[i]
                clase_pred   = nombres_clases[np.argmax(prob_imagen)]
                orden        = np.argsort(prob_imagen)[::-1]

                # Mostrar imagen
                imagen = to_pil_image(entradas[i].cpu())
                plt.imshow(imagen, cmap='gray')
                plt.axis('off')
                plt.title(f'Predicción: {clase_pred}')
                plt.show()

                # Mostrar probabilidades ordenadas
                print('Probabilidades por clase:')
                for idx in orden:
                    print(f'  {nombres_clases[idx]:<25} {prob_imagen[idx]*100:.2f}%')
                print('-' * 40)

                mostradas += 1


predecir_con_probabilidades(modelo, loader_test, NOMBRES_CLASES, max_imagenes=5)

## 9. Guardado del modelo

Guardamos el `state_dict` junto con metadatos del modelo para que el wrapper de inferencia pueda cargarlo correctamente.

In [ ]:
import os

# Ruta de guardado relativa al proyecto
RUTA_MODELO = os.path.join('..', 'models', 'model.pth')
os.makedirs(os.path.dirname(RUTA_MODELO), exist_ok=True)

# Guardar state_dict + metadatos
checkpoint = {
    'state_dict':   modelo.state_dict(),
    'num_clases':   3,
    'nombres_clases': NOMBRES_CLASES,
    'tamanio_imagen': TAMANIO_IMAGEN,
    'arquitectura': 'ModeloCNN'
}

torch.save(checkpoint, RUTA_MODELO)

# Verificar que el archivo se creó correctamente
tamanio_kb = os.path.getsize(RUTA_MODELO) / 1024
print(f'Modelo guardado en: {RUTA_MODELO}')
print(f'Tamaño del archivo: {tamanio_kb:.1f} KB')

# Verificación: cargar el modelo y hacer una predicción de prueba
print('\nVerificando carga del modelo...')
checkpoint_cargado = torch.load(RUTA_MODELO, map_location='cpu')
modelo_verificacion = ModeloCNN(num_clases=3)
modelo_verificacion.load_state_dict(checkpoint_cargado['state_dict'])
modelo_verificacion.eval()

tensor_prueba = torch.randn(1, 3, 100, 100)
with torch.no_grad():
    salida = modelo_verificacion(tensor_prueba)

print(f'Output shape: {list(salida.shape)}  ← esperado [1, 3]')
print('Modelo guardado y verificado correctamente.')